In [1]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
main_data = joblib.load("../data/01_data.joblib")

In [3]:
# Vérifier la définition de la cible, puis créer un split stratifié train 80 % / test 20 %.
data = main_data.copy()
target = data['Mental_Health_Risk']
data = data.drop(columns=["Student_ID" , "Age" , "Mental_Health_Risk"])

In [ ]:

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder , LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

x_train , x_test , y_train , y_test = train_test_split(data , target , test_size=0.2 , stratify=target , random_state=100)


nominal_features = ["Gender","Part_Time_Job","Previous_Mental_Health_History"]
ordinal_features = ["Year_of_Study","Physical_Activity","Diet_Quality"]
numerical_features = [
    "Sleep_Hours",
    "Study_Hours_Per_Week",
    "GPA",
    "Social_Media_Hours",
    "Screen_Time_Hours",
    "Financial_Stress",
    "Family_Support",
    "Attendance_Rate"
]


ordinal_categories = [
    [1, 2, 3, 4, 5],
    ["Low", "Moderate", "High"],
    ["Poor", "Average", "Good"]
]

preprocessing_nominal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessing_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessing_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessing_target = Pipeline([
    ("encoder" , LabelEncoder())
]), 

preprocessor = ColumnTransformer([
    ("nominal", preprocessing_nominal, nominal_features),
    ("ordinal", preprocessing_ordinal, ordinal_features),
    ("numerical", preprocessing_num, numerical_features),
])

x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

target_encoder = LabelEncoder() 
y_train_processed = target_encoder.fit_transform(y_train)
y_test_processed = target_encoder.transform(y_test)

In [ ]:
def iqr_outliers():
    for col in numerical_features : 
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
        display(outliers)
        count_outliers = len(outliers)
        print(f"Column: {col}, Outliers: {count_outliers}")

iqr_outliers()


In [ ]:
import joblib
joblib.dump({"x_train" : x_train_processed , "x_test" : x_test_processed , "y_train" : y_train_processed , "y_test" : y_test_processed} , "../data/02_data.joblib")